In [ ]:
import serial
import csv
import os
from datetime import datetime

# ===== CẤU HÌNH =====
SERIAL_PORT = "COM12"   # đổi nếu cần (COM3, COM5,...)
BAUD_RATE = 115200
CSV_FILE = "fire_dataset.csv"

# Header chuẩn
HEADER = ["timestamp", "temp", "humidity", "gas", "label"]

def file_exists_and_has_header(filename):
    if not os.path.exists(filename):
        return False
    
    with open(filename, 'r', newline='') as f:
        first_line = f.readline().strip()
        return first_line == ",".join(HEADER)

def ensure_csv_file():
    if not file_exists_and_has_header(CSV_FILE):
        print("Tạo file mới và ghi header...")
        with open(CSV_FILE, 'w', newline='') as f:
            writer = csv.writer(f)
            writer.writerow(HEADER)

def parse_line(line):
    parts = line.strip().split(",")

    if len(parts) != 5:
        return None

    timestamp, temp, humidity, gas, label = parts

    # Kiểm tra format timestamp
    try:
        datetime.strptime(timestamp, "%Y-%m-%d %H:%M:%S")
    except ValueError:
        return None

    try:
        temp = float(temp)
        humidity = float(humidity)
        gas = int(gas)
        label = int(label)
    except ValueError:
        return None

    return [timestamp, temp, humidity, gas, label]


def main():
    ensure_csv_file()

    print(f"Kết nối {SERIAL_PORT} ...")

    try:
        ser = serial.Serial(SERIAL_PORT, BAUD_RATE, timeout=1)
    except Exception as e:
        print("Không mở được cổng COM:", e)
        return

    print("Bắt đầu ghi dữ liệu... Nhấn Ctrl+C để dừng.")

    try:
        while True:
            line = ser.readline().decode("utf-8").strip()

            if not line:
                continue

            parsed = parse_line(line)

            if parsed:
                with open(CSV_FILE, 'a', newline='') as f:
                    writer = csv.writer(f)
                    writer.writerow(parsed)

                print("Đã ghi:", parsed)
            else:
                print("Bỏ qua dòng lỗi:", line)

    except KeyboardInterrupt:
        print("\nDừng chương trình.")
    finally:
        ser.close()


if __name__ == "__main__":
    main()